# Chapter 5 · Tokenization

A complete BPE tokenizer, trained on the text of chapters 1 to 4 of the book. Standard Python and a CPU: there is nothing to install. Run the cells in order.

The full explanation is in the PDF chapter. This notebook is the lab: the same program, split into parts, so you can try it with your own texts.

## 1. Characters, numbers and bytes

`ord` gives the Unicode number of each character, and `encode("utf-8")` turns it into bytes. Notice how many bytes the accented letters and the euro sign take.

In [1]:
for text in ["a", "é", "ñ", "€", "café", "La señal llegó"]:
    print(repr(text), [ord(c) for c in text], list(text.encode("utf-8")))

'a' [97] [97]
'é' [233] [195, 169]
'ñ' [241] [195, 177]
'€' [8364] [226, 130, 172]
'café' [99, 97, 102, 233] [99, 97, 102, 195, 169]
'La señal llegó' [76, 97, 32, 115, 101, 241, 97, 108, 32, 108, 108, 101, 103, 243] [76, 97, 32, 115, 101, 195, 177, 97, 108, 32, 108, 108, 101, 103, 195, 179]


## 2. Counting pairs and merging them

`count_pairs` counts how many times each pair of consecutive symbols appears. `merge` replaces a pair with a new symbol. In Colab, the first cell downloads the corpus if it is not there.

In [2]:
"""Chapter 5: a BPE tokenizer. Standard library and CPU only."""
from collections import Counter
from pathlib import Path

CORPUS = Path("material/corpus_chapters_1_4.txt")
if not CORPUS.exists():  # in Colab, download it from the repository
    import urllib.request
    CORPUS.parent.mkdir(exist_ok=True)
    urllib.request.urlretrieve("https://raw.githubusercontent.com/HernanDiaz/language-models/main/material/corpus_chapters_1_4.txt", CORPUS)
MERGES = 256
END = "<|end|>"


def count_pairs(ids):
    return Counter(zip(ids, ids[1:]))


def merge(ids, pair, new_id):
    result, i = [], 0
    while i < len(ids):
        if i + 1 < len(ids) and (ids[i], ids[i + 1]) == pair:
            result.append(new_id)
            i += 2
        else:
            result.append(ids[i])
            i += 1
    return result

## 3. Learning the merges

`train` repeats three steps: count, choose the most frequent pair and merge it. Check the four merges of the chapter's example by hand.

In [3]:
def train(text, merges=MERGES):
    ids = list(text.encode("utf-8"))
    rules = {}
    vocabulary = {i: bytes([i]) for i in range(256)}
    for k in range(merges):
        pairs = count_pairs(ids)
        if not pairs:
            break
        pair = max(pairs, key=pairs.get)
        if pairs[pair] < 2:
            break
        new_id = 256 + k
        ids = merge(ids, pair, new_id)
        rules[pair] = new_id
        vocabulary[new_id] = vocabulary[pair[0]] + vocabulary[pair[1]]
    return rules, vocabulary

In [4]:
rules4, voc4 = train("we bake we make we take", 4)
for pair, new_id in rules4.items():
    print(new_id, pair, repr(voc4[new_id].decode("utf-8")))

256 (101, 32) 'e '
257 (119, 256) 'we '
258 (97, 107) 'ak'
259 (258, 256) 'ake '


## 4. Encoding and decoding

`encode` applies the rules in the order in which they were learned; `decode` joins the bytes and recovers the text.

In [5]:
def encode(text, rules):
    ids = list(text.encode("utf-8"))
    for pair, new_id in rules.items():
        ids = merge(ids, pair, new_id)
    return ids


def decode(ids, vocabulary):
    return b"".join(vocabulary[i] for i in ids).decode("utf-8", errors="replace")


def pieces(ids, vocabulary):
    return [vocabulary[i].decode("utf-8", errors="replace") for i in ids]

In [6]:
for text in ["we make", "we bake cakes"]:
    ids = encode(text, rules4)
    print(repr(text), ids, pieces(ids, voc4), repr(decode(ids, voc4)))

rules_c, voc_c = train("café cafés", 3)
print("Token with half an é:", voc_c[258], "->", repr(voc_c[258].decode("utf-8", errors="replace")))

'we make' [257, 109, 258, 101] ['we ', 'm', 'ak', 'e'] 'we make'
'we bake cakes' [257, 98, 259, 99, 258, 101, 115] ['we ', 'b', 'ake ', 'c', 'ak', 'e', 's'] 'we bake cakes'
Token with half an é: b'caf\xc3' -> 'caf�'


## 5. Joining documents with a special token

In [7]:
def join_documents(documents, rules, end_id):
    ids = []
    for document in documents:
        ids.extend(encode(document, rules))
        ids.append(end_id)
    return ids

## 6. The complete experiment

It trains 256 merges on the book's corpus, encodes the whole corpus and checks the round trip. It takes about twenty seconds.

In [8]:
def run():
    text = CORPUS.read_text(encoding="utf-8")
    rules, vocabulary = train(text)
    end_id = len(vocabulary)
    vocabulary[end_id] = END.encode("utf-8")
    print("Vocabulary:", len(vocabulary), "tokens")
    new_ids = list(rules.values())
    print("First merges:", pieces(new_ids[:20], vocabulary))
    longest = sorted(new_ids, key=lambda i: len(vocabulary[i]), reverse=True)
    print("Longest tokens:", pieces(longest[:10], vocabulary))

    ids = encode(text, rules)
    n_bytes = len(text.encode("utf-8"))
    print("Corpus:", n_bytes, "bytes ->", len(ids), "tokens")
    print("Bytes per token: %.3f" % (n_bytes / len(ids)))
    print("Exact round trip:", decode(ids, vocabulary) == text)

    for phrase in ("La señal llegó", "probabilities", "strawberry",
                   "The network learns to count", "La red aprende a contar"):
        phrase_ids = encode(phrase, rules)
        print(repr(phrase), len(phrase.encode("utf-8")), "bytes ->",
              len(phrase_ids), "tokens:", pieces(phrase_ids, vocabulary))

    documents = join_documents(["ana", "bala"], rules, end_id)
    print("Two documents:", documents, repr(decode(documents, vocabulary)))
    print("The text <|end|> typed by someone:", encode(END, rules))
    return rules, vocabulary

In [9]:
rules, vocabulary = run()

Vocabulary: 513 tokens
First merges: ['e ', 'th', 's ', 't ', 'the ', 'in', 'er', ' a', ' the ', 'on', 'd ', '. ', 'ti', 'or', 'an', 're', 'en', 'ar', ' w', 'ro']
Longest tokens: ['probabilit', 'probabil', 'ing the ', 'of the ', 'e will ', 'es the ', 'network', 'in the ', 'program', '. The ']
Corpus: 117547 bytes -> 51680 tokens
Bytes per token: 2.275
Exact round trip: True
'La señal llegó' 16 bytes -> 11 tokens: ['L', 'a ', 'se', '�', '�', 'al ', 'll', 'e', 'g', '�', '�']
'probabilities' 13 bytes -> 3 tokens: ['probabil', 'iti', 'es']
'strawberry' 10 bytes -> 7 tokens: ['st', 'ra', 'w', 'b', 'er', 'r', 'y']
'The network learns to count' 27 bytes -> 9 tokens: ['The ', 'network', ' ', 'lear', 'n', 's ', 'to ', 'coun', 't']
'La red aprende a contar' 23 bytes -> 12 tokens: ['L', 'a ', 're', 'd', ' a', 'pre', 'n', 'd', 'e ', 'a ', 'cont', 'ar']
Two documents: [270, 97, 512, 98, 298, 97, 512] 'ana<|end|>bala<|end|>'
The text <|end|> typed by someone: [60, 124, 272, 100, 124, 62]


## 7. Try your own texts

Change the phrases in the list and see how the tokenizer splits them. Try long words, proper names, numbers and texts in other languages.

Look at the `é` of `café`: it takes two bytes that the tokenizer has not merged, because the book's text has no accented letters. When each token is shown separately, those bytes appear as the replacement character, although `decode` recovers the whole phrase without any problem.

In [10]:
for phrase in ["The tokenizer splits the text", "1234567", "We met at the café."]:
    ids = encode(phrase, rules)
    print(len(ids), "tokens:", pieces(ids, vocabulary), repr(decode(ids, vocabulary)))

14 tokens: ['The ', 'to', 'k', 'en', 'i', 'z', 'er ', 's', 'pl', 'its ', 'the ', 't', 'ex', 't'] 'The tokenizer splits the text'
7 tokens: ['1', '2', '3', '4', '5', '6', '7'] '1234567'
12 tokens: ['W', 'e ', 'me', 't ', 'at ', 'the ', 'c', 'a', 'f', '�', '�', '.'] 'We met at the café.'


## Exercises

1. Write the UTF-8 bytes of `naïve`. How many characters and how many bytes does it have? Why do they not match?

2. How many bytes do `résumé` and `RESUME` take? Explain the difference.

3. Repeat by hand the four merges of `we bake we make we take` and check the lengths in the chapter's table.

4. In the second merge there was a tie. Repeat the example choosing the other pair. After four merges, do the pieces into which the text is split change? And the numbers of the tokens?

5. Encode `make we` with the four rules, and then decode the identifiers you obtain.

6. Encode `team` with the four rules. Which rules apply? What happens to words that were not in the training text?

7. In the example `café cafés`, why can the token of the third merge not be shown as text? How does `decode` solve it?

8. Explain why a vocabulary of 256 bytes can represent any text. Relate it to the `rosa` problem of chapter 2.

9. According to the chapter's figure, how many bytes per token do we get with 128 and with 512 merges? Why does each new merge save less than the previous one?

10. Change the constant `MERGES` (line 6 of the program) to 64 and then to 512, and run it each time. Compare the number of tokens of the corpus and the longest tokens. With 512 merges the program takes somewhat longer, and `<|end|>` becomes token 768.

11. Why does the Spanish sentence need more tokens than the English one, even though it is shorter? What would happen with a tokenizer trained on Spanish texts?

12. Why is `<|end|>` added as an identifier and not as text? What happens if someone writes those characters in a message?

13. Segmentation analysis. Choose ten words: five that appear often in this book and five that never appear in it. Encode them with the chapter's tokenizer, write down their tokens in a table, check the round trip of each one and write five lines about what you observe.

## References

Structure based on [minbpe](https://github.com/karpathy/minbpe) and on the lecture [Let's build the GPT Tokenizer](https://www.youtube.com/watch?v=zduSFxRajkE), by Andrej Karpathy. The corpus, the examples and the explanations are original.